In [2]:
import pandas as pd

# Carrega os dados e converte a coluna de data
df = pd.read_csv('eventos.csv')
df['data_hora'] = pd.to_datetime(df['data_hora'])

# Correção: O formato padrão internacional é YYYY-MM-DD
dia_analise = '2026-10-28'
df_dia = df[df['data_hora'].dt.strftime('%Y-%m-%d') == dia_analise]

print(f"--- RELATÓRIO DO DIA {dia_analise} ---\n")

# 1. Quantas pessoas entraram na sala em determinado dia
df_entradas = df_dia[df_dia['resultado'] == 'ENTRADA']
total_entradas = df_entradas.shape[0]
pessoas_unicas_entraram = df_entradas['nome'].nunique()

print(f"1. Quantas pessoas entraram na sala: {pessoas_unicas_entraram} pessoa(s) (Total de {total_entradas} acessos de entrada).")

# 2. Quantas pessoas saíram da sala em determinado dia
df_saidas = df_dia[df_dia['resultado'] == 'SAIDA']
total_saidas = df_saidas.shape[0]
pessoas_unicas_sairam = df_saidas['nome'].nunique()

print(f"2. Quantas pessoas saíram da sala: {pessoas_unicas_sairam} pessoa(s) (Total de {total_saidas} registros de saída).")

# 3. Quanto tempo um colaborador específico permaneceu na sala em um determinado dia
nome_colaborador = 'Ana Souza'

df_colab = df_dia[(df_dia['nome'] == nome_colaborador) & 
                  (df_dia['resultado'].isin(['ENTRADA', 'SAIDA']))].sort_values('data_hora')

tempo_total_segundos = 0
hora_entrada = None

for index, row in df_colab.iterrows():
    if row['resultado'] == 'ENTRADA':
        hora_entrada = row['data_hora']
    elif row['resultado'] == 'SAIDA' and hora_entrada is not None:
        tempo_sessao = (row['data_hora'] - hora_entrada).total_seconds()
        tempo_total_segundos += tempo_sessao
        hora_entrada = None 

horas = int(tempo_total_segundos // 3600)
minutos = int((tempo_total_segundos % 3600) // 60)

print(f"\n3. Tempo total de permanência de '{nome_colaborador}': {horas} horas e {minutos} minutos (Total: {tempo_total_segundos} segundos).")

# 4. Quantas tentativas de acesso negado ocorreram
total_negados = df_dia[df_dia['resultado'] == 'NEGADO'].shape[0]
print(f"\n4. Total de tentativas de acesso negado: {total_negados}")

# 5. Quantas tentativas de invasão ocorreram
total_invasoes = df_dia[df_dia['resultado'] == 'INVASAO'].shape[0]
print(f"5. Total de tentativas de invasão (tags desconhecidas): {total_invasoes}")

# 6. Quais colaboradores não autorizados mais tentaram acessar a sala
print("\n6. Ranking de colaboradores não autorizados que mais tentaram acessar:")
ranking_negados = df_dia[df_dia['resultado'] == 'NEGADO']['nome'].value_counts()
print(ranking_negados.to_string())

--- RELATÓRIO DO DIA 2026-10-28 ---

1. Quantas pessoas entraram na sala: 3 pessoa(s) (Total de 10 acessos de entrada).
2. Quantas pessoas saíram da sala: 2 pessoa(s) (Total de 9 registros de saída).

3. Tempo total de permanência de 'Ana Souza': 0 horas e 0 minutos (Total: 0 segundos).

4. Total de tentativas de acesso negado: 6
5. Total de tentativas de invasão (tags desconhecidas): 4

6. Ranking de colaboradores não autorizados que mais tentaram acessar:
nome
Carlos    6
